In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

# ---- STEP 1: Load and chunk document ----
def load_and_chunk(filepath, chunk_size=400, overlap=50):
    with open(filepath, "r", encoding="utf-8") as file:
        raw_text = file.read()
    
    chunks = []
    start = 0
    while start < len(raw_text):
        
        end = start + chunk_size
        chunk = raw_text[start:end]
        if end < len(raw_text):
            last_period = chunk.rfind('.')
            if last_period > 100:
                end = start + last_period + 1
                chunk = raw_text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    
    return [c for c in chunks if len(c) > 30]

print("Loading documents...")
chunks = load_and_chunk("data/techmart_policy.txt")
print(f"Loaded {len(chunks)} chunks!")

In [ ]:
# ---- STEP 2: Set up embeddings and vector database ----
print("\nGenerating embeddings...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunks)

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="techmart_chat")
except:
    pass

collection = chroma_client.create_collection(name="techmart_chat")
ids = [f"chunk{i+1}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=ids
)

print(f"Vector database ready with {collection.count()} chunks!")

In [ ]:
# ---- STEP 3: Initialize Groq and conversation memory ----
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(dotenv_path=Path("../.env"), override=True)
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# This list is our conversation MEMORY
# It grows with every message exchanged
conversation_history = [
    {
        "role": "system",
        "content": """You are TechMart's helpful customer support assistant.
Answer questions using ONLY the context provided.
If the answer is not in the context, say 'I don't have that information.'
Always be polite and professional."""
    }
]

print("\n--- TechMart AI Chatbot ---")
print("Type 'quit' to exit\n")

# ---- STEP 4: The conversation loop ----
while True:
    # Get user input
    user_input = input("You: ")
    
    # Exit condition
    if user_input.lower() == "quit":
        print("Thank you for contacting TechMart. Goodbye!")
        break
    
    # Search vector database for relevant chunks
    question_embedding = embedder.encode([user_input]).tolist()
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )
    context = "\n".join(results['documents'][0])
    
    # Build prompt with context
    prompt = f"""Use ONLY this context to answer:

CONTEXT:
{context}

QUESTION: {user_input}"""

    # Add user message to conversation history
    conversation_history.append({
        "role": "user",
        "content": prompt
    })
    
    # Send FULL conversation history to LLM
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=conversation_history
    )
    
    # Get assistant reply
    assistant_reply = response.choices[0].message.content
    
    # Add assistant reply to conversation history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_reply
    })
    
    print(f"\nTechMart Bot: {assistant_reply}\n")

In [ ]:
os.getenv("GROQ_API_KEY")